# AirSense — Alert Logic

Notebook ini digunakan untuk mengembangkan rule-based alert logic yang menggabungkan informasi kondisi kualitas udara, anomaly detection, dan forecasting.

### Tujuan

1. Membedakan kondisi kualitas udara, anomali, dan peringatan.
2. Menggabungkan beberapa sumber evidence secara transparan.
3. Menentukan jenis dan tingkat alert yang dapat ditampilkan pada dashboard AirSense.
4. Menghindari pemberian alert hanya berdasarkan satu indikator tanpa konteks.

### Prinsip

- **ISPU** menunjukkan status kualitas udara berdasarkan perhitungan regulasi.
- **Anomaly detection** menunjukkan kondisi yang berbeda dari pola historis.
- **Forecasting** memperkirakan konsentrasi polutan 60 menit ke depan.
- **Alert** merupakan pesan kepada pengguna berdasarkan aturan yang telah ditentukan.

Anomali tidak secara otomatis berarti kondisi berbahaya, dan kondisi kualitas udara yang buruk tidak harus bersifat anomalous.

## 1. Desain Alert AirSense

Alert AirSense dibagi berdasarkan sumber informasi:

### Air Quality Alert
Dihasilkan ketika kondisi kualitas udara berdasarkan ISPU mencapai kategori yang membutuhkan perhatian.

### Anomaly Alert
Dihasilkan ketika terdapat evidence kuat bahwa kondisi sensor berbeda dari pola historis.

### Forecast Alert
Dihasilkan ketika hasil forecasting menunjukkan potensi memburuknya kondisi dalam 60 menit ke depan.

Setiap jenis alert dipertahankan secara terpisah agar pengguna dapat memahami alasan sebuah peringatan diberikan.

In [32]:
# Import library dan konfigurasi dasar

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.3f}"
)

# Parameter polutan utama AirSense
pollutant_cols = [
    "pm25_ugm3",
    "pm10_ugm3",
    "co_ugm3",
    "no2_ugm3",
    "o3_ugm3"
]

print("Parameter polutan:")
print(pollutant_cols)

Parameter polutan:
['pm25_ugm3', 'pm10_ugm3', 'co_ugm3', 'no2_ugm3', 'o3_ugm3']


## 2. Load Hasil Analytics

Alert logic menggunakan hasil dari tahap analytics sebelumnya. Pada tahap pengembangan, hasil anomaly detection yang telah disimpan digunakan sebagai input untuk membangun dan menguji aturan alert.

In [33]:
OUTPUT_DIR = Path("../outputs")

ANOMALY_PATH = (
    OUTPUT_DIR
    / "anomaly_detection_results.csv"
)

alert_df = pd.read_csv(
    ANOMALY_PATH,
    keep_default_na=False
)

alert_df["created_at"] = pd.to_datetime(
    alert_df["created_at"],
    utc=True
)

print("Jumlah data:", len(alert_df))

print(
    "\nDistribusi anomaly evidence:"
)

print(
    alert_df["anomaly_evidence"]
    .value_counts(dropna=False)
)

alert_df.head()

Jumlah data: 10081

Distribusi anomaly evidence:
anomaly_evidence
None                9405
Z-Score              575
Isolation Forest      71
Both                  30
Name: count, dtype: int64


,created_at,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,any_zscore_anomaly,zscore_anomaly_count,iforest_anomaly,iforest_score,anomaly_evidence
0,2026-08-30 13:49:00+00:00,11.310,14.870,2682.450,0.420,20.610,False,0,False,0.142,None
1,2026-08-30 13:50:00+00:00,16.180,21.710,2673.630,0.000,73.920,False,0,False,0.151,None
2,2026-08-30 13:51:00+00:00,15.330,17.590,2818.740,0.000,21.040,False,0,False,0.252,None
3,2026-08-30 13:52:00+00:00,13.200,9.970,2548.780,0.430,21.050,False,0,False,0.135,None
4,2026-08-30 13:53:00+00:00,16.700,17.530,2537.620,0.000,80.640,False,0,False,0.133,None


### Anomaly Alert

In [34]:
alert_df["anomaly_level"] = "None"

# Evidence satu metode
alert_df.loc[
    alert_df["anomaly_evidence"].isin(
        [
            "Z-Score",
            "Isolation Forest"
        ]
    ),
    "anomaly_level"
] = "Observation"

# Beberapa polutan anomalous secara bersamaan
alert_df.loc[
    alert_df["zscore_anomaly_count"] >= 2,
    "anomaly_level"
] = "Moderate"

# Kedua detector memberikan evidence
alert_df.loc[
    alert_df["anomaly_evidence"] == "Both",
    "anomaly_level"
] = "Strong"

alert_df[
    "anomaly_level"
].value_counts()

anomaly_level
None           9405
Observation     621
Strong           30
Moderate         25
Name: count, dtype: int64

### Apakah anomaly menghasilkan alert?

In [35]:
alert_df["anomaly_alert"] = (
    alert_df["anomaly_level"]
    .isin(
        [
            "Moderate",
            "Strong"
        ]
    )
)

Observation

→ disimpan untuk analytics/dashboard

→ TIDAK perlu mengganggu pengguna

Moderate / Strong


→ kandidat alert


In [36]:
anomaly_alert_summary = (
    alert_df[
        "anomaly_level"
    ]
    .value_counts()
    .rename_axis(
        "Anomaly Level"
    )
    .reset_index(
        name="Count"
    )
)

anomaly_alert_summary[
    "Percentage (%)"
] = (
    anomaly_alert_summary["Count"]
    / len(alert_df)
    * 100
).round(3)

anomaly_alert_summary

,Anomaly Level,Count,Percentage (%)
0,None,9405,93.294
1,Observation,621,6.160
2,Strong,30,0.298
3,Moderate,25,0.248


### Interpretasi Anomaly Alert

Dari 10.081 observasi, sebanyak 9.405 observasi (93,294%) tidak memiliki anomaly evidence yang memenuhi kriteria. Sebanyak 621 observasi (6,160%) dikategorikan sebagai Observation sehingga tetap dapat dicatat untuk kebutuhan analitik, tetapi tidak langsung menghasilkan alert kepada pengguna.

Sebanyak 25 observasi (0,248%) memiliki level Moderate karena terdapat minimal dua parameter yang terdeteksi anomalous secara bersamaan. Sementara itu, 30 observasi (0,298%) memiliki level Strong karena terdeteksi oleh Rolling Z-Score dan Isolation Forest.

Dengan aturan prototype ini, hanya level Moderate dan Strong yang menjadi kandidat anomaly alert. Pendekatan ini digunakan untuk mengurangi kemungkinan terlalu banyak peringatan akibat setiap perubahan statistik kecil.

Threshold dan aturan tersebut masih bersifat eksperimental karena dikembangkan menggunakan data dummy dan perlu dievaluasi kembali menggunakan data riil IoT.

## 3. Air Quality Alert Berdasarkan ISPU

Status kualitas udara diperlakukan secara terpisah dari anomaly detection. Nilai ISPU yang telah dihitung pada tahap sebelumnya digunakan untuk menentukan apakah kondisi kualitas udara saat ini memerlukan perhatian pengguna.

Data yang belum memiliki periode pengukuran yang cukup untuk menghasilkan ISPU tidak dianggap sebagai kategori Baik, melainkan dipertahankan sebagai data yang belum tersedia.

In [37]:
ISPU_PATH = (
    OUTPUT_DIR
    / "ispu_results.csv"
)

ispu_result = pd.read_csv(
    ISPU_PATH,
    keep_default_na=False
)

ispu_result["created_at"] = pd.to_datetime(
    ispu_result["created_at"],
    utc=True
)

# Kolom kosong dari CSV dikembalikan menjadi NaN
numeric_ispu_cols = [
    "pm25_ispu",
    "pm10_ispu",
    "co_ispu",
    "no2_ispu",
    "o3_ispu",
    "ispu_total"
]

for col in numeric_ispu_cols:
    ispu_result[col] = pd.to_numeric(
        ispu_result[col],
        errors="coerce"
    )

alert_df = alert_df.merge(
    ispu_result[
        [
            "created_at",
            "ispu_total",
            "dominant_pollutant",
            "ispu_category"
        ]
    ],
    on="created_at",
    how="left"
)

print(
    "Jumlah data setelah merge:",
    len(alert_df)
)

print(
    "\nDistribusi kategori ISPU:"
)

print(
    alert_df["ispu_category"]
    .replace("", "Belum tersedia")
    .value_counts(dropna=False)
)

alert_df[
    [
        "created_at",
        "ispu_total",
        "dominant_pollutant",
        "ispu_category",
        "anomaly_level"
    ]
].tail()

Jumlah data setelah merge: 10081

Distribusi kategori ISPU:
ispu_category
Sedang            4515
Baik              4127
Belum tersedia    1439
Name: count, dtype: int64


,created_at,ispu_total,dominant_pollutant,ispu_category,anomaly_level
10076,2026-09-06 13:45:00+00:00,51.000,PM2.5,Sedang,None
10077,2026-09-06 13:46:00+00:00,51.000,PM2.5,Sedang,None
10078,2026-09-06 13:47:00+00:00,51.000,PM2.5,Sedang,None
10079,2026-09-06 13:48:00+00:00,51.000,PM2.5,Sedang,None
10080,2026-09-06 13:49:00+00:00,51.000,PM2.5,Sedang,None


### Air Quality Alert

In [38]:
air_quality_alert_categories = [
    "Tidak Sehat",
    "Sangat Tidak Sehat",
    "Berbahaya"
]

alert_df["air_quality_alert"] = (
    alert_df["ispu_category"]
    .isin(
        air_quality_alert_categories
    )
)

print(
    alert_df[
        "air_quality_alert"
    ].value_counts()
)

air_quality_alert
False    10081
Name: count, dtype: int64


## 4. Forecast Alert

Forecast alert digunakan untuk memberikan informasi apabila hasil forecasting menunjukkan potensi memburuknya kondisi kualitas udara dalam 60 menit ke depan.

Forecast alert dipisahkan dari current air quality alert karena keduanya memiliki konteks waktu yang berbeda:

- current air quality menggambarkan kondisi berdasarkan data yang telah tersedia,
- forecast menggambarkan estimasi konsentrasi pada 60 menit berikutnya.

Pada tahap prototype, forecast alert dibangun dari hasil forecasting yang telah dikembangkan sebelumnya. Aturan ini masih bersifat eksperimental karena model masih menggunakan data dummy.

In [39]:
#Load Forecast
FORECAST_PATH = (
    OUTPUT_DIR
    / "forecast_t60_results.csv"
)

forecast_df = pd.read_csv(
    FORECAST_PATH
)

forecast_df["created_at"] = pd.to_datetime(
    forecast_df["created_at"],
    utc=True
)

forecast_df["forecast_at"] = pd.to_datetime(
    forecast_df["forecast_at"],
    utc=True
)

print(
    "Jumlah forecast:",
    len(forecast_df)
)

forecast_df.head()

Jumlah forecast: 1495


,created_at,pm25_ugm3_actual_t60,pm25_ugm3_forecast_t60,pm10_ugm3_actual_t60,pm10_ugm3_forecast_t60,co_ugm3_actual_t60,co_ugm3_forecast_t60,no2_ugm3_actual_t60,no2_ugm3_forecast_t60,o3_ugm3_actual_t60,o3_ugm3_forecast_t60,forecast_at
0,2026-09-05 11:55:00+00:00,17.400,15.107,23.320,17.961,2807.530,2857.654,0.000,0.000,79.200,52.442,2026-09-05 12:55:00+00:00
1,2026-09-05 11:56:00+00:00,8.000,15.307,9.430,18.159,2616.170,2847.596,0.000,0.000,20.840,52.184,2026-09-05 12:56:00+00:00
2,2026-09-05 11:57:00+00:00,13.410,15.315,15.000,17.903,2852.900,2854.200,0.000,0.000,19.950,49.124,2026-09-05 12:57:00+00:00
3,2026-09-05 11:58:00+00:00,12.930,15.676,14.700,19.111,2674.060,2853.542,0.580,0.000,57.800,42.705,2026-09-05 12:58:00+00:00
4,2026-09-05 11:59:00+00:00,13.100,15.739,18.740,18.499,2900.520,2777.879,0.000,0.073,20.670,43.900,2026-09-05 12:59:00+00:00


In [40]:
#Gungsi ISPU untuk Forecast
ISPU_BREAKPOINTS = {
    "pm25_ugm3": [
        (0, 15.5, 0, 50),
        (15.5, 55.4, 50, 100),
        (55.4, 150.4, 100, 200),
        (150.4, 250.4, 200, 300),
        (250.4, 500, 300, 500)
    ],
    "pm10_ugm3": [
        (0, 50, 0, 50),
        (50, 150, 50, 100),
        (150, 350, 100, 200),
        (350, 420, 200, 300),
        (420, 500, 300, 500)
    ],
    "co_ugm3": [
        (0, 4000, 0, 50),
        (4000, 8000, 50, 100),
        (8000, 15000, 100, 200),
        (15000, 30000, 200, 300),
        (30000, 45000, 300, 500)
    ],
    "o3_ugm3": [
        (0, 120, 0, 50),
        (120, 235, 50, 100),
        (235, 400, 100, 200),
        (400, 800, 200, 300),
        (800, 1000, 300, 500)
    ],
    "no2_ugm3": [
        (0, 80, 0, 50),
        (80, 200, 50, 100),
        (200, 1130, 100, 200),
        (1130, 2260, 200, 300),
        (2260, 3000, 300, 500)
    ]
}


def calculate_ispu(value, breakpoints):

    if pd.isna(value) or value < 0:
        return np.nan

    for c_low, c_high, i_low, i_high in breakpoints:

        if c_low <= value <= c_high:

            result = (
                (i_high - i_low)
                / (c_high - c_low)
                * (value - c_low)
                + i_low
            )

            return round(result)

    # Eksperimen untuk nilai di atas breakpoint tertinggi
    c_low, c_high, i_low, i_high = breakpoints[-1]

    result = (
        (i_high - i_low)
        / (c_high - c_low)
        * (value - c_low)
        + i_low
    )

    return round(result)

In [41]:
#Forecast Indicator
for pollutant in pollutant_cols:

    forecast_col = (
        f"{pollutant}_forecast_t60"
    )

    indicator_col = (
        f"{pollutant}_forecast_indicator"
    )

    forecast_df[indicator_col] = (
        forecast_df[forecast_col]
        .apply(
            lambda x: calculate_ispu(
                x,
                ISPU_BREAKPOINTS[pollutant]
            )
        )
    )

In [42]:
forecast_indicator_cols = [
    f"{col}_forecast_indicator"
    for col in pollutant_cols
]

forecast_df[
    "forecast_indicator_total"
] = forecast_df[
    forecast_indicator_cols
].max(axis=1)

forecast_df[
    "forecast_dominant_pollutant"
] = (
    forecast_df[
        forecast_indicator_cols
    ]
    .idxmax(axis=1)
    .str.replace(
        "_ugm3_forecast_indicator",
        "",
        regex=False
    )
    .str.upper()
)

forecast_df[
    [
        "created_at",
        "forecast_at",
        "forecast_indicator_total",
        "forecast_dominant_pollutant"
    ]
].head(10)

,created_at,forecast_at,forecast_indicator_total,forecast_dominant_pollutant
0,2026-09-05 11:55:00+00:00,2026-09-05 12:55:00+00:00,49,PM25
1,2026-09-05 11:56:00+00:00,2026-09-05 12:56:00+00:00,49,PM25
2,2026-09-05 11:57:00+00:00,2026-09-05 12:57:00+00:00,49,PM25
3,2026-09-05 11:58:00+00:00,2026-09-05 12:58:00+00:00,50,PM25
4,2026-09-05 11:59:00+00:00,2026-09-05 12:59:00+00:00,50,PM25
5,2026-09-05 12:00:00+00:00,2026-09-05 13:00:00+00:00,50,PM25
6,2026-09-05 12:01:00+00:00,2026-09-05 13:01:00+00:00,50,PM25
7,2026-09-05 12:02:00+00:00,2026-09-05 13:02:00+00:00,50,PM25
8,2026-09-05 12:03:00+00:00,2026-09-05 13:03:00+00:00,50,PM25
9,2026-09-05 12:04:00+00:00,2026-09-05 13:04:00+00:00,50,PM25


In [43]:
forecast_df[
    "forecast_indicator_total"
].describe()

count   1495.000
mean      50.566
std        5.132
min       41.000
25%       49.000
50%       50.000
75%       51.000
max       87.000
Name: forecast_indicator_total, dtype: float64

In [44]:
forecast_df[
    "forecast_dominant_pollutant"
].value_counts()

forecast_dominant_pollutant
PM25    1489
CO         6
Name: count, dtype: int64

In [45]:
forecast_df[
    [
        "created_at",
        "forecast_at",
        "forecast_indicator_total",
        "forecast_dominant_pollutant"
    ]
].head(10)

,created_at,forecast_at,forecast_indicator_total,forecast_dominant_pollutant
0,2026-09-05 11:55:00+00:00,2026-09-05 12:55:00+00:00,49,PM25
1,2026-09-05 11:56:00+00:00,2026-09-05 12:56:00+00:00,49,PM25
2,2026-09-05 11:57:00+00:00,2026-09-05 12:57:00+00:00,49,PM25
3,2026-09-05 11:58:00+00:00,2026-09-05 12:58:00+00:00,50,PM25
4,2026-09-05 11:59:00+00:00,2026-09-05 12:59:00+00:00,50,PM25
5,2026-09-05 12:00:00+00:00,2026-09-05 13:00:00+00:00,50,PM25
6,2026-09-05 12:01:00+00:00,2026-09-05 13:01:00+00:00,50,PM25
7,2026-09-05 12:02:00+00:00,2026-09-05 13:02:00+00:00,50,PM25
8,2026-09-05 12:03:00+00:00,2026-09-05 13:03:00+00:00,50,PM25
9,2026-09-05 12:04:00+00:00,2026-09-05 13:04:00+00:00,50,PM25


### Interpretasi Forecast Indicator

Hasil forecasting pada test set menghasilkan 1.495 prediksi untuk horizon 60 menit ke depan. Forecast indicator memiliki nilai antara 41 hingga 87, dengan median 50.

Sebagian besar parameter dominan pada forecast adalah PM2.5, yaitu 1.489 observasi, sedangkan CO menjadi parameter dominan pada 6 observasi.

Pada dataset eksperimen ini, forecast indicator belum mencapai nilai di atas 100. Oleh karena itu, hasil forecasting tidak menunjukkan estimasi kondisi pada rentang Tidak Sehat atau lebih tinggi.

Forecast indicator digunakan sebagai indikator eksperimental untuk mendukung pengembangan alert logic dan tidak diperlakukan sebagai ISPU resmi masa depan, karena perhitungan ISPU regulatif memiliki ketentuan agregasi konsentrasi yang perlu dipenuhi.

### Aturan Forecast Alert

Pada prototype ini, forecast alert diberikan apabila forecast indicator melebihi 100, yang menunjukkan bahwa estimasi kondisi 60 menit ke depan telah memasuki rentang yang memerlukan perhatian lebih tinggi.

Forecast indicator tidak diklaim sebagai ISPU resmi masa depan. Nilai ini digunakan sebagai indikator eksperimental untuk mendukung mekanisme early warning AirSense.

Perubahan nilai yang masih berada pada rentang Baik atau Sedang tetap dapat ditampilkan sebagai informasi tren, tetapi tidak langsung menghasilkan alert.

In [46]:
#Forecast Alert
forecast_df["forecast_alert"] = (
    forecast_df["forecast_indicator_total"] > 100
)

forecast_df[
    "forecast_alert"
].value_counts()

forecast_alert
False    1495
Name: count, dtype: int64

### Integrasikan Forecast ke alert_df

In [47]:
#Merge Forecast
forecast_merge_cols = [
    "created_at",
    "forecast_at",
    "forecast_indicator_total",
    "forecast_dominant_pollutant",
    "forecast_alert"
]

alert_df = alert_df.merge(
    forecast_df[
        forecast_merge_cols
    ],
    on="created_at",
    how="left"
)

print(
    "Jumlah data setelah merge forecast:",
    len(alert_df)
)

print(
    "\nForecast tersedia:",
    alert_df[
        "forecast_indicator_total"
    ].notna().sum()
)

print(
    "\nForecast alert:"
)

print(
    alert_df[
        "forecast_alert"
    ].value_counts(
        dropna=False
    )
)

Jumlah data setelah merge forecast: 10081

Forecast tersedia: 1495

Forecast alert:
forecast_alert
NaN      8586
False    1495
Name: count, dtype: int64


## 5. Unified Alert Logic

AirSense mempertahankan tiga sumber alert secara terpisah, yaitu kondisi kualitas udara saat ini, anomaly detection, dan forecasting.

Unified alert digunakan untuk menentukan apakah pada suatu timestamp terdapat informasi yang perlu mendapatkan perhatian pengguna. Jenis alert tetap disimpan agar alasan munculnya peringatan dapat dijelaskan secara transparan pada dashboard.

In [48]:
# NaN forecast berarti forecast tidak tersedia,
# bukan berarti forecast alert aktif.

forecast_alert_available = (
    alert_df["forecast_alert"]
    .fillna(False)
    .astype(bool)
)

alert_df["has_alert"] = (
    alert_df["air_quality_alert"]
    | alert_df["anomaly_alert"]
    | forecast_alert_available
)

alert_df[
    "has_alert"
].value_counts()

has_alert
False    10026
True        55
Name: count, dtype: int64

In [49]:
#Alert Type
def determine_alert_type(row):

    alert_types = []

    if row["air_quality_alert"]:
        alert_types.append(
            "Air Quality"
        )

    if row["anomaly_alert"]:
        alert_types.append(
            "Anomaly"
        )

    if pd.notna(row["forecast_alert"]):
        if row["forecast_alert"]:
            alert_types.append(
                "Forecast"
            )

    if not alert_types:
        return "None"

    return " + ".join(alert_types)


alert_df["alert_type"] = (
    alert_df.apply(
        determine_alert_type,
        axis=1
    )
)

alert_df[
    "alert_type"
].value_counts()

alert_type
None       10026
Anomaly       55
Name: count, dtype: int64

## 6. Alert Severity

Severity digunakan untuk menunjukkan tingkat perhatian dari alert yang dihasilkan AirSense.

Penentuan severity mempertimbangkan sumber alert secara terpisah. Kondisi kualitas udara memiliki prioritas berdasarkan kategori ISPU, sedangkan anomaly alert menggunakan tingkat evidence dari hasil anomaly detection. Forecast alert digunakan sebagai early warning ketika indikator forecast memasuki rentang yang membutuhkan perhatian.

Severity tidak digunakan untuk menyatakan bahwa setiap anomali merupakan kondisi berbahaya.

In [50]:
#Menentukan Severity
def determine_severity(row):

    severity_rank = {
        "None": 0,
        "Low": 1,
        "Medium": 2,
        "High": 3
    }

    candidates = ["None"]

    # -----------------------------
    # Air Quality
    # -----------------------------
    if row["air_quality_alert"]:

        if row["ispu_category"] == "Tidak Sehat":
            candidates.append("Medium")

        elif row["ispu_category"] in [
            "Sangat Tidak Sehat",
            "Berbahaya"
        ]:
            candidates.append("High")

    # -----------------------------
    # Anomaly
    # -----------------------------
    if row["anomaly_alert"]:

        if row["anomaly_level"] == "Moderate":
            candidates.append("Low")

        elif row["anomaly_level"] == "Strong":
            candidates.append("Medium")

    # -----------------------------
    # Forecast
    # -----------------------------
    if pd.notna(row["forecast_alert"]):

        if row["forecast_alert"]:
            candidates.append("Medium")

    return max(
        candidates,
        key=lambda x: severity_rank[x]
    )


alert_df["severity"] = alert_df.apply(
    determine_severity,
    axis=1
)

alert_df["severity"].value_counts()

severity
None      10026
Medium       30
Low          25
Name: count, dtype: int64

In [51]:
#Validasi Severity
pd.crosstab(
    alert_df["anomaly_level"],
    alert_df["severity"]
)

severity,Low,Medium,None
anomaly_level,,,
Moderate,25,0,0
None,0,0,9405
Observation,0,0,621
Strong,0,30,0


## 7. Alert Message

Setiap alert dilengkapi pesan singkat yang menjelaskan alasan peringatan muncul. Pesan dibuat secara rule-based agar sumber dan alasan alert tetap transparan serta dapat ditelusuri.

Pada tahap prototype, pesan tidak dihasilkan menggunakan generative AI.

In [52]:
#Message Generator
def generate_alert_message(row):

    messages = []

    # Air Quality Alert
    if row["air_quality_alert"]:

        messages.append(
            f"Kualitas udara berada pada kategori "
            f"{row['ispu_category']} dengan "
            f"parameter dominan "
            f"{row['dominant_pollutant']}."
        )

    # Anomaly Alert
    if row["anomaly_alert"]:

        if row["anomaly_level"] == "Strong":

            messages.append(
                "Perubahan tidak biasa terdeteksi "
                "oleh dua metode anomaly detection."
            )

        elif row["anomaly_level"] == "Moderate":

            messages.append(
                "Perubahan tidak biasa terdeteksi "
                "pada beberapa parameter kualitas udara."
            )

    # Forecast Alert
    if (
        pd.notna(row["forecast_alert"])
        and row["forecast_alert"]
    ):

        messages.append(
            f"Forecast menunjukkan potensi peningkatan "
            f"kondisi kualitas udara dalam 60 menit "
            f"dengan parameter dominan "
            f"{row['forecast_dominant_pollutant']}."
        )

    if not messages:
        return ""

    return " ".join(messages)


alert_df["alert_message"] = alert_df.apply(
    generate_alert_message,
    axis=1
)

In [53]:
#Periksa Alert yang Dihasilkan
alert_examples = alert_df.loc[
    alert_df["has_alert"],
    [
        "created_at",
        "ispu_category",
        "anomaly_level",
        "forecast_indicator_total",
        "alert_type",
        "severity",
        "alert_message"
    ]
]

print(
    "Jumlah alert:",
    len(alert_examples)
)

alert_examples.head(20)

Jumlah alert: 55


,created_at,ispu_category,anomaly_level,forecast_indicator_total,alert_type,severity,alert_message
82,2026-08-30 15:11:00+00:00,,Moderate,NaN,Anomaly,Low,Perubahan tidak biasa terdeteksi pada beberapa...
620,2026-08-31 00:09:00+00:00,,Moderate,NaN,Anomaly,Low,Perubahan tidak biasa terdeteksi pada beberapa...
1118,2026-08-31 08:27:00+00:00,,Moderate,NaN,Anomaly,Low,Perubahan tidak biasa terdeteksi pada beberapa...
2105,2026-09-01 00:54:00+00:00,Baik,Moderate,NaN,Anomaly,Low,Perubahan tidak biasa terdeteksi pada beberapa...
2259,2026-09-01 03:28:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...
2333,2026-09-01 04:42:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...
2334,2026-09-01 04:43:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...
2335,2026-09-01 04:44:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...
2336,2026-09-01 04:45:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...
2337,2026-09-01 04:46:00+00:00,Baik,Strong,NaN,Anomaly,Medium,Perubahan tidak biasa terdeteksi oleh dua meto...


In [54]:
#Cek Distribusinya
alert_summary = (
    alert_df.loc[
        alert_df["has_alert"]
    ]
    .groupby(
        [
            "alert_type",
            "severity"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)

alert_summary

,alert_type,severity,Count
0,Anomaly,Low,25
1,Anomaly,Medium,30


In [55]:
#Analisis Jarak Antar Alert
alert_times = (
    alert_df.loc[
        alert_df["has_alert"],
        ["created_at"]
    ]
    .sort_values("created_at")
    .copy()
)

alert_times["minutes_since_previous"] = (
    alert_times["created_at"]
    .diff()
    .dt.total_seconds()
    / 60
)

alert_times.head(20)

,created_at,minutes_since_previous
82,2026-08-30 15:11:00+00:00,NaN
620,2026-08-31 00:09:00+00:00,538.000
1118,2026-08-31 08:27:00+00:00,498.000
2105,2026-09-01 00:54:00+00:00,987.000
2259,2026-09-01 03:28:00+00:00,154.000
2333,2026-09-01 04:42:00+00:00,74.000
2334,2026-09-01 04:43:00+00:00,1.000
2335,2026-09-01 04:44:00+00:00,1.000
2336,2026-09-01 04:45:00+00:00,1.000
2337,2026-09-01 04:46:00+00:00,1.000


In [56]:
alert_times[
    "minutes_since_previous"
].describe()

count     54.000
mean     180.556
std      331.399
min        1.000
25%        1.000
50%        8.500
75%      169.750
max     1547.000
Name: minutes_since_previous, dtype: float64

In [57]:
print(
    "Alert berjarak <= 5 menit:",
    (
        alert_times[
            "minutes_since_previous"
        ] <= 5
    ).sum()
)

print(
    "Alert berjarak > 5 menit:",
    (
        alert_times[
            "minutes_since_previous"
        ] > 5
    ).sum()
)

Alert berjarak <= 5 menit: 25
Alert berjarak > 5 menit: 29


## 8. Alert Event Grouping

Data AirSense diperbarui setiap satu menit sehingga satu kejadian anomali dapat menghasilkan alert pada beberapa timestamp yang berdekatan. Mengirim setiap timestamp sebagai notifikasi terpisah dapat menyebabkan alert berulang untuk kejadian yang sama.

Pada prototype ini, alert yang memiliki jarak maksimum 5 menit dari alert sebelumnya dikelompokkan ke dalam satu event. Jika jaraknya lebih dari 5 menit, timestamp tersebut dianggap sebagai awal event baru.

Threshold 5 menit merupakan parameter desain awal berdasarkan pola pada data eksperimen dan perlu dievaluasi kembali menggunakan data sensor riil.

In [58]:
# Membentuk event berdasarkan jarak antar-alert

ALERT_EVENT_GAP_MINUTES = 5

alert_events = (
    alert_df.loc[
        alert_df["has_alert"]
    ]
    .sort_values("created_at")
    .copy()
)

alert_events["minutes_since_previous"] = (
    alert_events["created_at"]
    .diff()
    .dt.total_seconds()
    / 60
)

alert_events["new_event"] = (
    alert_events["minutes_since_previous"].isna()
    | (
        alert_events["minutes_since_previous"]
        > ALERT_EVENT_GAP_MINUTES
    )
)

alert_events["alert_event_id"] = (
    alert_events["new_event"]
    .cumsum()
)

print(
    "Jumlah timestamp alert:",
    len(alert_events)
)

print(
    "Jumlah alert event:",
    alert_events["alert_event_id"].nunique()
)

Jumlah timestamp alert: 55
Jumlah alert event: 30


In [59]:
#Ringkasan per Event
severity_rank = {
    "None": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3
}

alert_events["severity_rank"] = (
    alert_events["severity"]
    .map(severity_rank)
)

event_summary = (
    alert_events
    .groupby("alert_event_id")
    .agg(
        start_time=("created_at", "min"),
        end_time=("created_at", "max"),
        duration_records=("created_at", "size"),
        max_severity_rank=("severity_rank", "max")
    )
    .reset_index()
)

rank_to_severity = {
    0: "None",
    1: "Low",
    2: "Medium",
    3: "High"
}

event_summary["max_severity"] = (
    event_summary["max_severity_rank"]
    .map(rank_to_severity)
)

event_summary["duration_minutes"] = (
    (
        event_summary["end_time"]
        - event_summary["start_time"]
    )
    .dt.total_seconds()
    / 60
)

event_summary.head(20)

,alert_event_id,start_time,end_time,duration_records,max_severity_rank,max_severity,duration_minutes
0,1,2026-08-30 15:11:00+00:00,2026-08-30 15:11:00+00:00,1,1,Low,0.000
1,2,2026-08-31 00:09:00+00:00,2026-08-31 00:09:00+00:00,1,1,Low,0.000
2,3,2026-08-31 08:27:00+00:00,2026-08-31 08:27:00+00:00,1,1,Low,0.000
3,4,2026-09-01 00:54:00+00:00,2026-09-01 00:54:00+00:00,1,1,Low,0.000
4,5,2026-09-01 03:28:00+00:00,2026-09-01 03:28:00+00:00,1,2,Medium,0.000
5,6,2026-09-01 04:42:00+00:00,2026-09-01 04:50:00+00:00,8,2,Medium,8.000
6,7,2026-09-01 04:57:00+00:00,2026-09-01 04:57:00+00:00,1,2,Medium,0.000
7,8,2026-09-01 05:05:00+00:00,2026-09-01 05:05:00+00:00,1,2,Medium,0.000
8,9,2026-09-01 08:24:00+00:00,2026-09-01 08:24:00+00:00,1,1,Low,0.000
9,10,2026-09-01 08:33:00+00:00,2026-09-01 08:38:00+00:00,2,1,Low,5.000


In [60]:
#Distribusi Event
print(
    "Jumlah event:",
    len(event_summary)
)

print(
    "\nDistribusi severity event:"
)

print(
    event_summary[
        "max_severity"
    ].value_counts()
)

print(
    "\nStatistik jumlah timestamp per event:"
)

print(
    event_summary[
        "duration_records"
    ].describe()
)

print(
    "\nStatistik durasi event (menit):"
)

print(
    event_summary[
        "duration_minutes"
    ].describe()
)

Jumlah event: 30

Distribusi severity event:
max_severity
Low       18
Medium    12
Name: count, dtype: int64

Statistik jumlah timestamp per event:
count   30.000
mean     1.833
std      2.086
min      1.000
25%      1.000
50%      1.000
75%      1.000
max      8.000
Name: duration_records, dtype: float64

Statistik durasi event (menit):
count   30.000
mean     1.033
std      2.385
min      0.000
25%      0.000
50%      0.000
75%      0.000
max      8.000
Name: duration_minutes, dtype: float64


In [61]:
#Ambil pesan representatif / buat event message
representative_alerts = (
    alert_events
    .sort_values(
        [
            "alert_event_id",
            "severity_rank"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop_duplicates(
        subset="alert_event_id",
        keep="first"
    )
    [
        [
            "alert_event_id",
            "alert_type",
            "alert_message"
        ]
    ]
)

event_summary = event_summary.merge(
    representative_alerts,
    on="alert_event_id",
    how="left"
)

event_summary[
    [
        "alert_event_id",
        "start_time",
        "end_time",
        "duration_minutes",
        "duration_records",
        "max_severity",
        "alert_type",
        "alert_message"
    ]
].head(20)

,alert_event_id,start_time,end_time,duration_minutes,duration_records,max_severity,alert_type,alert_message
0,1,2026-08-30 15:11:00+00:00,2026-08-30 15:11:00+00:00,0.000,1,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...
1,2,2026-08-31 00:09:00+00:00,2026-08-31 00:09:00+00:00,0.000,1,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...
2,3,2026-08-31 08:27:00+00:00,2026-08-31 08:27:00+00:00,0.000,1,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...
3,4,2026-09-01 00:54:00+00:00,2026-09-01 00:54:00+00:00,0.000,1,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...
4,5,2026-09-01 03:28:00+00:00,2026-09-01 03:28:00+00:00,0.000,1,Medium,Anomaly,Perubahan tidak biasa terdeteksi oleh dua meto...
5,6,2026-09-01 04:42:00+00:00,2026-09-01 04:50:00+00:00,8.000,8,Medium,Anomaly,Perubahan tidak biasa terdeteksi oleh dua meto...
6,7,2026-09-01 04:57:00+00:00,2026-09-01 04:57:00+00:00,0.000,1,Medium,Anomaly,Perubahan tidak biasa terdeteksi oleh dua meto...
7,8,2026-09-01 05:05:00+00:00,2026-09-01 05:05:00+00:00,0.000,1,Medium,Anomaly,Perubahan tidak biasa terdeteksi oleh dua meto...
8,9,2026-09-01 08:24:00+00:00,2026-09-01 08:24:00+00:00,0.000,1,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...
9,10,2026-09-01 08:33:00+00:00,2026-09-01 08:38:00+00:00,5.000,2,Low,Anomaly,Perubahan tidak biasa terdeteksi pada beberapa...


In [62]:
#Simpan Output
# Menambahkan event ID kembali ke data utama

event_mapping = (
    alert_events[
        [
            "created_at",
            "alert_event_id"
        ]
    ]
)

alert_df = alert_df.merge(
    event_mapping,
    on="created_at",
    how="left"
)

# Simpan detail alert per timestamp
alert_df.to_csv(
    OUTPUT_DIR / "alert_results.csv",
    index=False
)

# Simpan ringkasan event
event_summary.to_csv(
    OUTPUT_DIR / "alert_events.csv",
    index=False
)

print(
    "Output alert berhasil disimpan."
)

print(
    "Timestamp:",
    len(alert_df)
)

print(
    "Timestamp dengan alert:",
    alert_df["has_alert"].sum()
)

print(
    "Jumlah alert event:",
    len(event_summary)
)

Output alert berhasil disimpan.
Timestamp: 10081
Timestamp dengan alert: 55
Jumlah alert event: 30


## Kesimpulan

Prototype alert logic AirSense berhasil mengintegrasikan tiga sumber informasi, yaitu kondisi kualitas udara, anomaly detection, dan forecasting.

Pada dataset dummy yang digunakan, tidak ditemukan Air Quality Alert maupun Forecast Alert karena kondisi dan hasil forecast belum mencapai threshold yang ditentukan. Sebanyak 55 timestamp menghasilkan Anomaly Alert, terdiri dari 25 alert dengan severity Low dan 30 alert dengan severity Medium.

Setelah dilakukan event grouping dengan gap maksimum 5 menit, 55 timestamp alert tersebut dikelompokkan menjadi 30 alert event. Sebanyak 18 event memiliki severity Low dan 12 event memiliki severity Medium. Pengelompokan ini membantu mengurangi kemungkinan notifikasi berulang ketika satu kejadian anomali terdeteksi pada beberapa timestamp yang berdekatan.

Anomaly detection diperlakukan sebagai evidence perubahan pola dan tidak secara otomatis menunjukkan kondisi udara berbahaya. Forecast indicator juga merupakan indikator eksperimental dan tidak diklaim sebagai ISPU resmi masa depan.

Seluruh threshold, severity, dan aturan event grouping pada tahap ini masih merupakan rancangan prototype berbasis data dummy dan perlu dievaluasi kembali menggunakan data sensor riil.